# Tutorial - Legacy converter

## 1. Installation

We use the PyPi package from the [Antares legacy-to-GEMS converter repository](https://github.com/AntaresSimulatorTeam/AntaresLegacyModels-to-GEMS-Converter)

In [3]:
import subprocess

# Install the Antares-to-GEMS converter
subprocess.run(
    ["uv", "pip", "install", "antares-legacy-gems-converter"],
    check=True,
)

Using Python 3.12.3 environment at: /home/gmaistre/Documents/GEMS/GEMS/.venv
Checked 1 package in 62ms


CompletedProcess(args=['uv', 'pip', 'install', 'antares-legacy-gems-converter'], returncode=0)

**Restart the kernel** after running the cell above so the newly installed package is picked up.

## 2. Running converter

### 2.1 Introduction of the sample legacy study

The legacy study which is about to be conveted is composed of two areas (`france`, `germany`) with one thermal plant per country and one wind plant in `france`.

| Element  |Power/unit|  Unit    |  Marginal Price   |
|:----------|:----------:|:----------:|:----------:|
| france_thermal_gas    | 10 MW    | 10 units    | 45€ / MW    |
| france_wind_onshore    | 60 MW    | 1 unit    | ***   |
| germany_thermal_gas    | 30 MW    | 5 units    | 60€ / MW    |


In [4]:
from pathlib import Path
from antares.craft import read_study_local

print("STUDY LOADING")
_cwd = Path.cwd()

# Parse the Antares legacy study into a Study object
study = read_study_local(_cwd / "notebook_legacy")
print("\tStudy loaded")

STUDY LOADING
	Study loaded


### 2.2 Set the converter's output folder
The folder of where the converted study will be is set as `tmp/converter_output`.

In [5]:
import shutil

# Set the output directory
output_path = _cwd / "tmp" /"converter_output"
if output_path.exists():
    shutil.rmtree(output_path)

print("\tOutput path set")

	Output path set


### 2.3 Set the library used by the converter
Converter uses the `antares-legacy-models` library for adapting Antares legacy structure elements to GEMS models language.

In [6]:
# Path to the antares_legacy_models.yml defining the GEMS component models that each
# Antares concept (area, thermal, load, hydro …) maps to.
lib_path = (_cwd / "../../../libraries/antares_legacy_models.yml").resolve()
print("\tLibrary path resolved")

	Library path resolved


### 2.4 Running the conversion
The configuration is now set, the conversion can start.

In [ ]:
from antares_gems_converter.input_converter.src.converter import AntaresStudyConverter
from antares_gems_converter.input_converter.src.logger import Logger

# Set logger
logger = Logger("notebook_test", study.path)

print("\nCONVERSION STARTING")

# Build the converter
# mode="full" : generate a standalone GEMS study written from scratch.
# models_to_convert lists every model type present in notebook_legacy.
converter = AntaresStudyConverter(
    study_input=study,
    logger=logger,
    mode="full",
    output_folder=output_path,
    lib_paths=[str(lib_path)],
    models_to_convert=["load", "wind", "thermal",
                       ],
    )

print("\tConverter set")
# Run the full conversion pipeline and write the result to:
#   output_folder/<study_name>/input/system.yml
converter.process_all()
print("\tConversion succeeded")

print(f"Saved to: {converter.output_system_path}")

2026/07/15 14:25:27 logger,line: 41     INFO | Logger object created successfully.. 

CONVERSION STARTING
2026/07/15 14:25:27 converter,line: 100  WARNING | 'area' is not in models_to_convert but conversion mode is 'full': adding it automatically.
	Converter set
2026/07/15 14:25:27 converter,line: 483     INFO | Converting components of model load...
2026/07/15 14:25:27 converter,line: 483     INFO | Converting components of model thermal...
2026/07/15 14:25:28 converter,line: 483     INFO | Converting components of model area...
2026/07/15 14:25:28 converter,line: 548     INFO | Dumping input system into yaml file...
	Conversion succeeded
Saved to: /home/gmaistre/Documents/GEMS/GEMS/doc/getting-started/tutorial-four-antares-legacy/tmp/converter_output/notebook_legacy/input/system.yml


## 3. Running the converted study with GemsPy

Likewise in the notebooks about the [unit-commitment](../tutorial-one-unit-commitment/tutorial-unit-commitment.ipynb) and the [investment](../tutorial-three-investment/tutorial-invest.ipynb), the Python GEMS interpreter, GemsPy, is used for running the study in this notebook.

### 3.1 Installation of the required libraries

In [8]:
import subprocess, os
subprocess.run(
    ["uv", "sync", "--group", "doc", "--frozen"],
    check=True, capture_output=True,
    cwd=os.path.abspath(os.path.join(os.getcwd(), "../../.."))
)

CompletedProcess(args=['uv', 'sync', '--group', 'doc', '--frozen'], returncode=0, stdout=b'', stderr=b'\x1bUninstalled \x1b8 packages\x1b \x1bin 234ms\x1b\x1b\n\x1bInstalled \x1b1 package\x1b \x1bin 141ms\x1b\x1b\n \x1b-\x1b \x1bantares-craft\x1b\x1b==0.14.0\x1b\n \x1b-\x1b \x1bantares-legacy-gems-converter\x1b\x1b==0.2.1\x1b\n \x1b-\x1b \x1bantares-study-version\x1b\x1b==1.0.20\x1b\n \x1b-\x1b \x1bantares-timeseries-generation\x1b\x1b==0.1.9\x1b\n \x1b-\x1b \x1bpandas\x1b\x1b==2.3.3\x1b\n \x1b+\x1b \x1bpandas\x1b\x1b==3.0.3\x1b\n \x1b-\x1b \x1bpyarrow\x1b\x1b==25.0.0\x1b\n \x1b-\x1b \x1bpytz\x1b\x1b==2026.2\x1b\n \x1b-\x1b \x1btzdata\x1b\x1b==2026.3\x1b\n')

### 3.2 Loading and Solving the converted study

In [ ]:
import yaml
from pathlib import Path
from gems.study.folder import load_study
from gems.session.session import SimulationSession
from gems.optim_config.parsing import OptimConfig, TimeScopeConfig, ScenarioScopeConfig

study_dir = output_path / study.name

print("STUDY LOADING")
study = load_study(study_dir)
print("\tStudy loaded")

optim_config = OptimConfig(
    time_scope=TimeScopeConfig(first_time_step=0, last_time_step=167),
    scenario_scope=ScenarioScopeConfig(include=[0, 1, 2]),
)

print("\nSOLVING OPTIMIZATION PROBLEM")
result = SimulationSession(study=study, optim_config=optim_config).run()
print("\tOptimization problem solved")

SOLVING OPTIMIZATION PROBLEM


ValueError: An error occurred during parsing: 3 validation errors for LibrarySchema
port-types.0.area-connection
  Extra inputs are not permitted [type=extra_forbidden, input_value={'injection-to-balance': ...ied-energy-bound': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
version
  Extra inputs are not permitted [type=extra_forbidden, input_value='2.1.2', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
taxonomy
  Extra inputs are not permitted [type=extra_forbidden, input_value='antares_legacy_taxonomy', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden

We can print the objective value found for this study

In [ ]:
obj_value = float(result.data[result.data["output"] == "objective-value"]["value"].iloc[0])
print(f"Objective value: {obj_value}")

FileNotFoundError: No result found in /home/gmaistre/Documents/GEMS/GEMS/doc/getting-started/tutorial-four-antares-legacy/tmp/converter_output/notebook_legacy/output